In [ ]:
import os
import pickle
from copy import deepcopy
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric as tg
from sklearn.metrics import r2_score
from torch.utils.data import DataLoader

ROOT = Path(".").resolve()
DATA_DIR = ROOT / "model_data"
OUT_DIR = ROOT / "models"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("torch:", torch.__version__)
print("torch_geometric:", tg.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
class GAT(nn.Module):

    def __init__(self, num_layers, in_shape, out_shape, hidden_shape,
                 attn_heads=1, no_norm=False):
        super().__init__()
        if num_layers < 1:
            raise ValueError("num_layers must be >= 1")
        if attn_heads < 1:
            raise ValueError("attn_heads must be >= 1")

        self.out_dim = out_shape
        self.no_norm = no_norm
        self.activation = nn.LeakyReLU()
        self.conv_layers = nn.ModuleList()
        self.norms = nn.ModuleList()

        if num_layers == 1:
            self.conv_layers.append(
                tg.nn.GATv2Conv(in_shape, out_shape, heads=1, concat=False)
            )
            self.norms.append(nn.LayerNorm(out_shape))
        else:
            self.conv_layers.append(
                tg.nn.GATv2Conv(
                    in_shape, hidden_shape, heads=attn_heads, concat=True
                )
            )
            self.norms.append(nn.LayerNorm(hidden_shape * attn_heads))
            in_h = hidden_shape * attn_heads
            for _ in range(num_layers - 2):
                self.conv_layers.append(
                    tg.nn.GATv2Conv(
                        in_h, hidden_shape, heads=attn_heads, concat=True
                    )
                )
                self.norms.append(nn.LayerNorm(hidden_shape * attn_heads))
                in_h = hidden_shape * attn_heads
            self.conv_layers.append(
                tg.nn.GATv2Conv(
                    in_h, out_shape, heads=attn_heads, concat=False
                )
            )
            self.norms.append(nn.LayerNorm(out_shape))

    def forward(self, h, adj):
        last = len(self.conv_layers) - 1
        for i, (conv, norm) in enumerate(zip(self.conv_layers, self.norms)):
            h = conv(h, adj)
            h = self.activation(h)
            if not self.no_norm:
                h = norm(h)
                h = F.normalize(h, dim=1)
        return h


class Predictor(nn.Module):

    def __init__(self, in_size, layer_size, layer_num):
        super().__init__()
        self.input_layer = nn.Linear(in_size, layer_size)
        self.hidden_layers = nn.ModuleList(
            [nn.Linear(layer_size, layer_size) for _ in range(layer_num)]
        )
        self.output_layer = nn.Linear(layer_size, 1)
        self.activation = nn.ReLU()
        self.norms = nn.ModuleList(
            [nn.LayerNorm(layer_size) for _ in range(layer_num + 1)]
        )

    def forward(self, x):
        x = self.norms[0](self.activation(self.input_layer(x)))
        for i, layer in enumerate(self.hidden_layers):
            x = self.norms[i + 1](self.activation(layer(x)))
        return self.output_layer(x)


class RMSELoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.mse = nn.MSELoss()
        self.eps = eps

    def forward(self, pred, target):
        return torch.sqrt(self.mse(pred, target) + self.eps)


def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)


print("GAT + custom-GNN norms defined")


In [ ]:
@dataclass
class Config:
    node_data_len: int = 33
    edge_dim: int = 4  
    seed: int = 29

    num_layers: int = 2
    hidden_shape: int = 128
    out_shape: int = 64
    attn_heads: int = 1
    no_norm: bool = False
    predictor_layer_size: int = 64
    predictor_layer_num: int = 4

    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    epochs: int = 200
    accumulation_steps: int = 16
    lr_step_size: int = 10
    lr_gamma: float = 0.99
    min_delta: float = 1e-5

    device: str = field(
        default_factory=lambda: "cuda:0" if torch.cuda.is_available() else "cpu"
    )

    use_wandb: bool = False


SWEEP_CONFIG = {
    "method": "random",
    "metric": {"name": "val_rmse", "goal": "minimize"},
    "parameters": {
        "learning_rate": {
            "distribution": "log_uniform_values", "min": 0.0000001, "max": 0.1
        },
        "batch_size": {
            "distribution": "q_log_uniform_values", "q": 2, "min": 1, "max": 40
        },
        "predictor_layer_size": {"values": [64, 96, 128]},
        "attn_heads": {"values": [1]},
        "num_layers": {
            "values": [1, 2]
        },
        "out_shape": {
            "distribution": "q_log_uniform_values", "q": 2, "min": 8, "max": 64
        },
        "hidden_shape": {
            "distribution": "q_log_uniform_values", "q": 2, "min": 32, "max": 192
        },
    },
}



def config_from_wandb(wc, base):
    return replace(
        base,
        learning_rate=float(wc.learning_rate),
        accumulation_steps=int(wc.batch_size),
        num_layers=max(1, int(wc.num_layers)),
        out_shape=max(8, int(wc.out_shape)),
        hidden_shape=max(8, int(wc.hidden_shape)),
        predictor_layer_size=int(wc.predictor_layer_size),
        predictor_layer_num=4,
        attn_heads=max(1, int(wc.attn_heads)),
        use_wandb=True,
    )


def _wandb_log(cfg, data):
    if cfg.use_wandb:
        import wandb
        wandb.log(data)


def load_graph_splits(cfg):
    data_dir = DATA_DIR
    splits = {
        name: np.load(data_dir / f"{name}.npy", allow_pickle=True)
        for name in ("train", "val", "test")
    }
    print(
        f"[data] predefined split -> train={len(splits['train'])} "
        f"val={len(splits['val'])} test={len(splits['test'])}"
    )
    return splits["train"], splits["val"], splits["test"]


def load_heat_scaler(cfg):
    with open(DATA_DIR / "scalers.pkl", "rb") as f:
        scalers = pickle.load(f)
    print("[data] loaded train-only target scaler for physical-unit RMSE")
    return scalers["target_scaler"]


def unpack_unit(unit, cfg, device):
    node = torch.as_tensor(unit["nodes"], dtype=torch.float32).reshape(
        -1, cfg.node_data_len
    )
    if node.shape[0] <= 1:
        return None
    adj = torch.as_tensor(unit["adj"], dtype=torch.long).reshape(2, -1)
    y = torch.as_tensor(unit["y"], dtype=torch.float32).reshape(-1)
    return node.to(device), adj.to(device), y.to(device)


def build_models(cfg):
    device = torch.device(cfg.device)
    model = GAT(
        num_layers=cfg.num_layers,
        in_shape=cfg.node_data_len,
        out_shape=cfg.out_shape,
        hidden_shape=cfg.hidden_shape,
        attn_heads=cfg.attn_heads,
        no_norm=cfg.no_norm,
    ).to(device)
    predictor = Predictor(
        in_size=model.out_dim,
        layer_size=cfg.predictor_layer_size,
        layer_num=cfg.predictor_layer_num,
    ).to(device)
    model.apply(init_weights)
    predictor.apply(init_weights)
    return model, predictor


@torch.no_grad()
def evaluate(model, predictor, loader, cfg, loss_func):
    device = torch.device(cfg.device)
    model.eval()
    predictor.eval()
    preds, targets = [], []
    for unit in loader:
        parsed = unpack_unit(unit, cfg, device)
        if parsed is None:
            continue
        node, adj, y = parsed
        out = predictor(model(node, adj)).flatten()
        preds.append(out.cpu())
        targets.append(y.cpu())
    preds = torch.cat(preds)
    targets = torch.cat(targets)
    rmse = loss_func(preds, targets).item()
    r2 = r2_score(targets.numpy(), preds.numpy())
    return r2, rmse, preds.numpy(), targets.numpy()


print("configuration and data pipeline defined")


In [ ]:
def train(cfg):
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    device = torch.device(cfg.device)
    print(f"[setup] device = {device}")

    train_set, val_set, test_set = load_graph_splits(cfg)
    heat_scaler = load_heat_scaler(cfg)

    train_loader = DataLoader(train_set, batch_size=1, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=1, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=1, shuffle=False)

    model, predictor = build_models(cfg)
    params = list(model.parameters()) + list(predictor.parameters())
    n_params = sum(p.numel() for p in params if p.requires_grad)
    print(f"[setup] trainable parameters = {n_params}")
    _wandb_log(cfg, {"parameter_count": n_params})

    loss_func = RMSELoss()
    optimizer = torch.optim.Adam(
        params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=cfg.lr_step_size, gamma=cfg.lr_gamma
    )

    train_curve, val_curve = [], []
    best_val = float("inf")
    best_state = None

    for epoch in range(cfg.epochs):
        model.train()
        predictor.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_losses = []
        seen_in_batch = 0

        for unit in train_loader:
            parsed = unpack_unit(unit, cfg, device)
            if parsed is None:
                continue
            node, adj, y = parsed

            out = predictor(model(node, adj)).flatten()
            loss = loss_func(out, y)
            (loss / cfg.accumulation_steps).backward()

            epoch_losses.append(loss.item())
            seen_in_batch += 1
            if seen_in_batch == cfg.accumulation_steps:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                seen_in_batch = 0

        if seen_in_batch > 0:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        scheduler.step()
        train_rmse = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
        val_r2, val_rmse, _, _ = evaluate(
            model, predictor, val_loader, cfg, loss_func
        )
        train_curve.append(train_rmse)
        val_curve.append(val_rmse)
        print(
            f"[epoch {epoch + 1:02d}/{cfg.epochs}] "
            f"lr={scheduler.get_last_lr()[0]:.2e} "
            f"train_rmse={train_rmse:.4f} val_rmse={val_rmse:.4f} "
            f"val_r2={val_r2:.4f}"
        )
        _wandb_log(cfg, {
            "epoch": epoch,
            "lr": scheduler.get_last_lr()[0],
            "train_rmse": train_rmse,
            "val_rmse": val_rmse,
            "val_r2": val_r2,
        })

        if val_rmse < best_val - cfg.min_delta:
            best_val = val_rmse
            best_state = {
                "model": {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                },
                "predictor": {
                    k: v.detach().cpu().clone()
                    for k, v in predictor.state_dict().items()
                },
            }

    if best_state is not None:
        model.load_state_dict(best_state["model"])
        predictor.load_state_dict(best_state["predictor"])

    post_training(
        cfg, model, predictor, loss_func, heat_scaler,
        train_loader, val_loader, test_loader, train_curve, val_curve
    )
    return model, predictor


def _physical_rmse(scaler, preds, targets):
    p = scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
    t = scaler.inverse_transform(targets.reshape(-1, 1)).flatten()
    return float(np.sqrt(np.mean((p - t) ** 2)))


def post_training(cfg, model, predictor, loss_func, heat_scaler,
                  train_loader, val_loader, test_loader, train_curve, val_curve):
    print("\n[post] final evaluation on best checkpoint")
    test_r2 = test_rmse = test_rmse_phys = float("nan")
    for name, loader in (
        ("train", train_loader), ("val", val_loader), ("test", test_loader)
    ):
        r2, rmse, preds, targets = evaluate(
            model, predictor, loader, cfg, loss_func
        )
        phys = _physical_rmse(heat_scaler, preds, targets)
        print(f"  {name:5s}: R2={r2:.4f} RMSE={rmse:.4f} rmse(phys)={phys:.3f}")
        _wandb_log(cfg, {
            f"{name}_r2_final": r2,
            f"{name}_rmse_final": rmse,
            f"{name}_rmse_phys": phys,
        })
        if name == "test":
            test_r2, test_rmse, test_rmse_phys = r2, rmse, phys

    model_path = os.path.join(
        str(OUT_DIR), f"gat_R2_{test_r2:.4f}_RMSE_{test_rmse:.4f}.pt"
    )
    pred_path = os.path.join(
        str(OUT_DIR), f"mlp_R2_{test_r2:.4f}_RMSE_{test_rmse:.4f}.pt"
    )
    torch.save(model.state_dict(), model_path)
    torch.save(predictor.state_dict(), pred_path)
    print(f"[post] saved model     -> {model_path}")
    print(f"[post] saved predictor -> {pred_path}")

    ckpt_path = os.path.join(
        str(OUT_DIR), f"checkpoint_gat_R2_{test_r2:.4f}_RMSE_{test_rmse:.4f}.pt"
    )
    torch.save({
        "model_state": model.state_dict(),
        "predictor_state": predictor.state_dict(),
        "config": asdict(cfg),
        "metrics": {"test_r2": test_r2, "test_rmse": test_rmse, "test_rmse_phys": test_rmse_phys},
        "architecture": "GAT_norms",
    }, ckpt_path)
    print(f"[post] saved full checkpoint -> {ckpt_path}")

    plt.figure(figsize=(7, 4))
    plt.plot(range(1, len(train_curve) + 1), train_curve, label="train RMSE")
    plt.plot(range(1, len(val_curve) + 1), val_curve, label="val RMSE")
    plt.xlabel("epoch")
    plt.ylabel("RMSE")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(str(OUT_DIR), "loss_curve.png"), dpi=120)
    plt.show()


print("training pipeline defined")


In [ ]:
def train_sweep():
    import wandb
    wandb.init()
    cfg = config_from_wandb(wandb.config, Config(use_wandb=True))
    try:
        wandb.config.update(
            {f"cfg/{k}": v for k, v in asdict(cfg).items()},
            allow_val_change=True,
        )
        train(cfg)
    finally:
        wandb.finish()


def run_sweep(count=20, project="", entity=None):
    import wandb
    sweep_id = wandb.sweep(SWEEP_CONFIG, project=project, entity=entity)
    wandb.agent(sweep_id, function=train_sweep, count=count)



In [ ]:
import wandb
wandb.login()
# hyperparameter search can be done here with Wandb
run_sweep(count=20, project="", entity="")


In [ ]:
def load_checkpoint(path, device=None, no_norm=None):
    ckpt = torch.load(path, map_location="cpu")
    if "config" not in ckpt or "model_state" not in ckpt:
        raise ValueError(
            "Expected a bundled checkpoint containing config, model_state, "
            "and predictor_state"
        )

    cfg = Config(**ckpt["config"])
    if no_norm is not None:
        cfg = replace(cfg, no_norm=no_norm)
    dev = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
    cfg = replace(cfg, device=dev)

    model, predictor = build_models(cfg)
    model.load_state_dict(ckpt["model_state"], strict=not cfg.no_norm)
    predictor.load_state_dict(ckpt["predictor_state"])
    model.eval()
    predictor.eval()
    return model, predictor, cfg


def plot_regression(preds, targets, title, scaler=None, save_path=None):
    preds = np.asarray(preds).reshape(-1)
    targets = np.asarray(targets).reshape(-1)
    if scaler is not None:
        preds = scaler.inverse_transform(preds.reshape(-1, 1)).ravel()
        targets = scaler.inverse_transform(targets.reshape(-1, 1)).ravel()
        xlabel, ylabel = "Predicted (phys)", "Ground truth (phys)"
    else:
        xlabel, ylabel = "Predicted (scaled)", "Ground truth (scaled)"

    lo = float(min(preds.min(), targets.min()))
    hi = float(max(preds.max(), targets.max()))

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(preds, targets, s=12, alpha=0.45)
    ax.plot([lo, hi], [lo, hi], linestyle="--", color="green", label="x = y")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.25)
    ax.set_aspect("equal", adjustable="box")
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=120)
        print(f"[reproduce] saved plot -> {save_path}")
    plt.show()
    return fig


def reproduce_from_checkpoint(path, device=None, plot=True, physical=True,
                              save_plots=True, no_norm=False):
    model, predictor, cfg = load_checkpoint(path, device, no_norm=no_norm)
    train_set, val_set, test_set = load_graph_splits(cfg)
    heat_scaler = load_heat_scaler(cfg)
    loss_func = RMSELoss()

    loaders = {
        "train": DataLoader(train_set, batch_size=1),
        "val": DataLoader(val_set, batch_size=1),
        "test": DataLoader(test_set, batch_size=1),
    }

    print(f"[reproduce] loaded {path}")
    print(f"[reproduce] evaluating on device={cfg.device}")
    results = {}
    plot_dir = None
    if plot and save_plots:
        plot_dir = Path(str(OUT_DIR)) / "reproduce_plots"
        plot_dir.mkdir(parents=True, exist_ok=True)

    for name, loader in loaders.items():
        r2, rmse, preds, targets = evaluate(
            model, predictor, loader, cfg, loss_func
        )
        phys = _physical_rmse(heat_scaler, preds, targets)
        print(f"  {name:5s}: R2={r2:.4f} RMSE={rmse:.4f} rmse(phys)={phys:.3f}")
        results[name] = {
            "r2": r2,
            "rmse": rmse,
            "rmse_phys": phys,
            "preds": preds,
            "targets": targets,
        }

        if plot:
            unit = "phys" if physical else "scaled"
            title = f"{name}  R2={r2:.4f}  RMSE={rmse:.4f}"
            if physical:
                title += f"  RMSE(phys)={phys:.3f}"
            save_path = None
            if plot_dir is not None:
                save_path = str(plot_dir / f"{name}_regression_{unit}.png")
            plot_regression(
                preds,
                targets,
                title=title,
                scaler=heat_scaler if physical else None,
                save_path=save_path,
            )

    return results, model, predictor, cfg


In [ ]:
reproduce_from_checkpoint("./models/checkpoint_gat_R2_0.9262_RMSE_0.2792.pt", 
save_plots=False, no_norm=False)

In [ ]:
reproduce_from_checkpoint("./models/checkpoint_gat_R2_0.9298_RMSE_0.2722.pt",no_norm=True)